# Medalian Architechture

In [0]:
data = [
    (1,"Ram","Laptop",1200),
    (2,"Sam","Mobile",800),
    (3,None,"Tablet",500),
    (4,"John","Laptop",-100),
    (5,"Ram","Mobile",800),
    (5,"Ram","Mobile",800)
]

columns = ["id","customer","product","amount"]

df = spark.createDataFrame(data,columns)

df.show()


In [0]:
bronze_path = "/Volumes/workspace/delta/medallion/bronze/sales"

df.write \
.format("delta") \
.mode("overwrite") \
.save(bronze_path)


In [0]:
bronze_table=spark.read.format("delta").load(bronze_path)
bronze_table.show()

In [0]:
silver_table = bronze_table.dropDuplicates()
silver_table = silver_table.fillna({"customer":"unknown"})
silver_table = silver_table.filter("amount > 0")
silver_table = silver_table.withColumnRenamed("id","sale_id")


In [0]:
display(silver_table)

In [0]:
gold_table = silver_table.groupBy("product").agg({"amount":"sum"})
gold_table = gold_table.withColumnRenamed("sum(amount)","total_sales")
gold_table = gold_table.orderBy("total_sales",ascending=False)
display(gold_table)